In [2]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
import pickle
from scipy.stats import norm

In [ ]:
# =========================
# LOAD DATA
# =========================
df = pd.read_csv("../nifty_final_dataset.csv")
df = df.dropna().reset_index(drop=True)

# =========================
# RETURNS (in %)
# =========================
returns = df["target"].values * 100

# =========================
# GARCH(1,1) LOG LIKELIHOOD
# =========================
def garch_loglik(params, returns):
    omega, alpha, beta = params
    
    T = len(returns)
    var = np.zeros(T)
    
    # initialize variance
    var[0] = np.var(returns)
    
    for t in range(1, T):
        var[t] = omega + alpha * returns[t-1]**2 + beta * var[t-1]
    
    # avoid zero variance
    var = np.clip(var, 1e-6, None)
    
    loglik = -0.5 * np.sum(
        np.log(2 * np.pi) +
        np.log(var) +
        (returns**2) / var
    )
    
    return -loglik  # minimize

# =========================
# INITIAL GUESS
# =========================
init_params = [0.1, 0.1, 0.8]

bounds = [
    (1e-6, 1),   # omega
    (0, 1),      # alpha
    (0, 1)       # beta
]

# =========================
# FIT MODEL
# =========================
result = minimize(
    garch_loglik,
    init_params,
    args=(returns,),
    bounds=bounds
)

omega, alpha, beta = result.x

print("\n===== TRAINED PARAMETERS =====")
print(f"Omega : {omega:.6f}")
print(f"Alpha : {alpha:.6f}")
print(f"Beta  : {beta:.6f}")

# =========================
# SAVE MODEL
# =========================
model = {
    "omega": omega,
    "alpha": alpha,
    "beta": beta
}

with open("garch_scratch.pkl", "wb") as f:
    pickle.dump(model, f)

print("\nModel saved successfully!")


===== TRAINED PARAMETERS =====
Omega : 0.031177
Alpha : 0.126971
Beta  : 0.845270

Model saved successfully!


In [3]:

# =========================
# LOAD DATA + MODEL
# =========================
df = pd.read_csv("../nifty_final_dataset.csv")
df = df.dropna().reset_index(drop=True)

returns = df["target"].values * 100

with open("garch_scratch.pkl", "rb") as f:
    model = pickle.load(f)

omega = model["omega"]
alpha = model["alpha"]
beta  = model["beta"]

# =========================
# COMPUTE LAST VARIANCE
# =========================
T = len(returns)
var = np.zeros(T)
var[0] = np.var(returns)

for t in range(1, T):
    var[t] = omega + alpha * returns[t-1]**2 + beta * var[t-1]

last_var = var[-1]
last_return = returns[-1]

# =========================
# NEXT DAY VARIANCE
# =========================
next_var = omega + alpha * last_return**2 + beta * last_var
next_vol = np.sqrt(next_var)

print("\n===== NEXT DAY FORECAST =====")
print(f"Volatility (σ): {next_vol:.4f}%")

# =========================
# POINT FORECAST (mean ≈ 0)
# =========================
point_forecast = 0

# =========================
# CONFIDENCE INTERVALS
# =========================
conf_levels = [0.90, 0.95, 0.99]

print("\n===== CONFIDENCE INTERVALS =====")

for conf in conf_levels:
    z = norm.ppf((1 + conf) / 2)
    
    lower = point_forecast - z * next_vol
    upper = point_forecast + z * next_vol
    
    print(f"{int(conf*100)}% CI: [{lower:.4f}%, {upper:.4f}%]")

# =========================
# OPTIONAL PRICE FORECAST
# =========================
last_price = df["close"].iloc[-1]

lower_price = last_price * (1 + lower / 100)
upper_price = last_price * (1 + upper / 100)

print("\n===== PRICE RANGE =====")
print(f"Expected Range: {lower_price:.2f} - {upper_price:.2f}")


===== NEXT DAY FORECAST =====
Volatility (σ): 1.7543%

===== CONFIDENCE INTERVALS =====
90% CI: [-2.8855%, 2.8855%]
95% CI: [-3.4383%, 3.4383%]
99% CI: [-4.5187%, 4.5187%]

===== PRICE RANGE =====
Expected Range: 21788.46 - 23850.74
